# Embeddings Retrieval

This notebook implements the embeddings-based retrieval pipeline:
1. Encode documents and queries into dense vectors.
2. Inspect embedding structure and shapes.
3. Visualize embeddings in 2D (UMAP/t-SNE).
4. Retrieve top-k documents per query with cosine similarity.


## Why High-Dimensional Embeddings?

High-dimensional embeddings map text into vector spaces where semantic similarity is preserved.
Compared to simpler lexical models, embeddings can better capture:
- synonymy (different words, similar meaning),
- contextual similarity,
- broader semantic relationships beyond exact token overlap.

In retrieval, this helps return relevant documents even when query wording and document wording are different.


In [ ]:
import sys
import json
import time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(project_root))

from src.data.load import load_all
from src.data.preprocess import add_content_field
from src.retrieval.embeddings import build_embeddings, retrieve_embeddings
from src.evaluation.evaluate import adapt_ground_truth, evaluate_run


In [ ]:
# Configuration :
K_VALUES = [5, 10, 20, 50]
PRIMARY_K = 20  # k used to rank configurations in the benchmark tables.
TEXT_FIELD = "content"

# Single-run baseline (used in the first part of the notebook).
MODEL_NAME = "all-MiniLM-L12-v2"
BATCH_SIZE = 64

RAW_DIR = project_root / "data" / "raw"
PROCESSED_DIR = project_root / "data" / "processed"
CACHE_DIR = project_root / "data" / "cache"

# Train controls.
# Default to a reproducible subset for notebook runtime. Set to None for the full corpus.
MAX_DOCS = 5000               # None for full corpus, else e.g. 5000 / 20000.
MAX_QUERIES = None            # None for all train queries, else e.g. 100 / 200.
FILTER_QUERIES_WITH_RELEVANT_IN_SUBSET = True
RANDOM_SEED = 42

PLOT_MAX_DOCS = 1200
BENCHMARK_SHOW_PROGRESS_BAR = False  # Keep False for repeated runs.
LOCAL_FILES_ONLY = False             # Cache is still used first; set True for strict HF local-only mode.

DEFAULT_EMBEDDING_DEVICE = "auto"   # "auto" | None | "cpu" | "mps" | "cuda"

def _resolve_embedding_device_spec(device: str | None):
    """Return a safe device value for SentenceTransformers (`None` means auto)."""
    if device in (None, "auto"):
        return None

    try:
        import torch
    except Exception:
        print(f"[warn] torch unavailable; falling back to auto device instead of {device!r}.")
        return None

    if device == "cuda" and not torch.cuda.is_available():
        print("[warn] CUDA requested but unavailable; falling back to auto device.")
        return None

    has_mps = bool(getattr(torch.backends, "mps", None)) and torch.backends.mps.is_available()
    if device == "mps" and not has_mps:
        print("[warn] MPS requested but unavailable; falling back to auto device.")
        return None

    return device

RESOLVED_DEFAULT_EMBEDDING_DEVICE = _resolve_embedding_device_spec(DEFAULT_EMBEDDING_DEVICE)

# Grid of embedding experiments for pure bi-encoder benchmark.
# (mpnet removed by default because it is too slow in your environment)
EMBEDDING_EXPERIMENTS = [
    {
        "name": "L12_quality",
        "model_name": "all-MiniLM-L12-v2",
        "batch_size": 64,
        "device": DEFAULT_EMBEDDING_DEVICE,
        "precision": "float32",
        "model_max_seq_length": None,
        "truncate_dim": None,
        "chunk_size": None,
        "normalize_embeddings": True,
        "local_files_only": LOCAL_FILES_ONLY,
    },
    {
        "name": "L12_balanced_128",
        "model_name": "all-MiniLM-L12-v2",
        "batch_size": 128,
        "device": DEFAULT_EMBEDDING_DEVICE,
        "precision": "float32",
        "model_max_seq_length": 128,
        "truncate_dim": 384,
        "chunk_size": None,
        "normalize_embeddings": True,
        "local_files_only": LOCAL_FILES_ONLY,
    },
    {
        "name": "L12_fast_96",
        "model_name": "all-MiniLM-L12-v2",
        "batch_size": 256,
        "device": DEFAULT_EMBEDDING_DEVICE,
        "precision": "int8",
        "model_max_seq_length": 96,
        "truncate_dim": 256,
        "chunk_size": None,
        "normalize_embeddings": True,
        "local_files_only": LOCAL_FILES_ONLY,
    },
]

# RRF4 candidate sources (fixed 4-source pool):
# 2 embedding sources + BM25 plus + BM25 okapi
RERANK_CANDIDATE_SOURCES = [
    {"name": "emb_L12_quality", "type": "embedding", "embedding_experiment": "L12_quality", "weight": 3.5},
    {"name": "emb_L12_balanced", "type": "embedding", "embedding_experiment": "L12_balanced_128", "weight": 1.0},
    {"name": "bm25_plus", "type": "bm25", "bm25_method": "plus", "weight": 0.5},
    {"name": "bm25_okapi", "type": "bm25", "bm25_method": "okapi", "weight": 0.5},
]

# Automatic RRF4 sweep on train.
# Each config can override: candidate_multiplier, rrf_k, and per-source weights.
RRF4_SWEEP_CONFIGS = [
    {
        "name": "rrf4_base_m1_k20",
        "candidate_multiplier": 1,
        "rrf_k": 20,
        "weights": {
            "emb_L12_quality": 3.5,
            "emb_L12_balanced": 1.0,
            "bm25_plus": 0.5,
            "bm25_okapi": 0.5,
        },
    },
    {
        "name": "rrf4_m1_k15",
        "candidate_multiplier": 1,
        "rrf_k": 15,
        "weights": {
            "emb_L12_quality": 3.5,
            "emb_L12_balanced": 1.0,
            "bm25_plus": 0.5,
            "bm25_okapi": 0.5,
        },
    },
    {
        "name": "rrf4_m1_k25",
        "candidate_multiplier": 1,
        "rrf_k": 25,
        "weights": {
            "emb_L12_quality": 3.5,
            "emb_L12_balanced": 1.0,
            "bm25_plus": 0.5,
            "bm25_okapi": 0.5,
        },
    },
    {
        "name": "rrf4_embA4_bm2505",
        "candidate_multiplier": 1,
        "rrf_k": 20,
        "weights": {
            "emb_L12_quality": 4.0,
            "emb_L12_balanced": 1.0,
            "bm25_plus": 0.5,
            "bm25_okapi": 0.5,
        },
    },
    {
        "name": "rrf4_embA3_bm2505",
        "candidate_multiplier": 1,
        "rrf_k": 20,
        "weights": {
            "emb_L12_quality": 3.0,
            "emb_L12_balanced": 1.0,
            "bm25_plus": 0.5,
            "bm25_okapi": 0.5,
        },
    },
    {
        "name": "rrf4_low_balanced",
        "candidate_multiplier": 1,
        "rrf_k": 20,
        "weights": {
            "emb_L12_quality": 3.5,
            "emb_L12_balanced": 0.3,
            "bm25_plus": 0.5,
            "bm25_okapi": 0.5,
        },
    },
    {
        "name": "rrf4_more_bm25",
        "candidate_multiplier": 1,
        "rrf_k": 20,
        "weights": {
            "emb_L12_quality": 3.5,
            "emb_L12_balanced": 1.0,
            "bm25_plus": 0.7,
            "bm25_okapi": 0.7,
        },
    },
    {
        "name": "rrf4_candidate_x2",
        "candidate_multiplier": 2,
        "rrf_k": 20,
        "weights": {
            "emb_L12_quality": 3.5,
            "emb_L12_balanced": 1.0,
            "bm25_plus": 0.5,
            "bm25_okapi": 0.5,
        },
    },
]

print("project_root:", project_root)
print("Model (baseline):", MODEL_NAME)
print("k values:", K_VALUES)
print("PRIMARY_K:", PRIMARY_K)
print("MAX_DOCS:", MAX_DOCS)
print("MAX_QUERIES:", MAX_QUERIES)
print("FILTER_QUERIES_WITH_RELEVANT_IN_SUBSET:", FILTER_QUERIES_WITH_RELEVANT_IN_SUBSET)
print("n_embedding_experiments:", len(EMBEDDING_EXPERIMENTS))
print("LOCAL_FILES_ONLY:", LOCAL_FILES_ONLY)
print("DEFAULT_EMBEDDING_DEVICE:", DEFAULT_EMBEDDING_DEVICE)
print("RESOLVED_DEFAULT_EMBEDDING_DEVICE:", RESOLVED_DEFAULT_EMBEDDING_DEVICE or "auto")
print("n_candidate_sources:", len(RERANK_CANDIDATE_SOURCES))
print("n_rrf4_sweep_configs:", len(RRF4_SWEEP_CONFIGS))


## Data Preparation

The dataset in this notebook includes a unified `content` field for both documents and queries.
This is the required text input format for embedding-based retrieval.


In [ ]:
# Load project data with the shared content field + train ground truth.
docs_path = PROCESSED_DIR / "docs_with_content.json"
queries_path = PROCESSED_DIR / "queries_train_with_content.json"
gts_path = RAW_DIR / "qgts_train.json"

if docs_path.exists() and queries_path.exists():
    with open(docs_path, "r", encoding="utf-8") as f:
        docs = json.load(f)
    with open(queries_path, "r", encoding="utf-8") as f:
        queries = json.load(f)
    print("Loaded processed docs/train queries with content.")
else:
    docs_raw, queries_raw, _, _ = load_all(RAW_DIR)
    docs, queries = add_content_field(docs_raw, queries_raw, clean=True)
    print("Processed files not found. Rebuilt content from raw files.")

with open(gts_path, "r", encoding="utf-8") as f:
    gts_raw = json.load(f)
gt_full = adapt_ground_truth(gts_raw)

# Build a deterministic subset for notebook speed.
if MAX_DOCS is not None and len(docs) > MAX_DOCS:
    category_to_docs = defaultdict(list)
    for d in docs:
        category_to_docs[str(d.get("category", "unknown"))].append(d)

    categories = sorted(category_to_docs.keys())
    per_category = max(1, MAX_DOCS // max(1, len(categories)))

    selected = []
    selected_ids = set()
    for cat in categories:
        for d in category_to_docs[cat][:per_category]:
            did = str(d.get("id"))
            if did not in selected_ids:
                selected.append(d)
                selected_ids.add(did)

    if len(selected) < MAX_DOCS:
        for d in docs:
            did = str(d.get("id"))
            if did in selected_ids:
                continue
            selected.append(d)
            selected_ids.add(did)
            if len(selected) >= MAX_DOCS:
                break

    docs = selected[:MAX_DOCS]

# Optional deterministic query sub-sampling.
rng = np.random.default_rng(RANDOM_SEED)
if MAX_QUERIES is not None and len(queries) > MAX_QUERIES:
    idx = np.sort(rng.choice(len(queries), size=MAX_QUERIES, replace=False))
    queries = [queries[i] for i in idx]

# Align queries with ground truth and with the current doc subset.
doc_id_set = set(str(d.get("id")) for d in docs)
queries_eval = []
gt_eval: dict[str, list[str]] = {}

for q in queries:
    qid = str(q.get("id"))
    if qid not in gt_full:
        continue

    rel_in_subset = [doc_id for doc_id in gt_full[qid] if doc_id in doc_id_set]
    if FILTER_QUERIES_WITH_RELEVANT_IN_SUBSET and len(rel_in_subset) == 0:
        continue

    queries_eval.append(q)
    gt_eval[qid] = rel_in_subset

queries = queries_eval
query_ids = [str(q.get("id")) for q in queries]

if not queries:
    raise ValueError(
        "No queries left after filtering. Increase MAX_DOCS or disable "
        "FILTER_QUERIES_WITH_RELEVANT_IN_SUBSET."
    )

print(f"docs={len(docs)}, queries_eval={len(queries)}, gt_eval={len(gt_eval)}")
doc_categories = pd.Series([str(d.get("category", "unknown")) for d in docs])
print("n_doc_categories:", doc_categories.nunique())
print(doc_categories.value_counts().head(10))

relevant_counts = pd.Series([len(gt_eval[qid]) for qid in query_ids])
print("relevant_docs_per_query (subset) - mean:", round(float(relevant_counts.mean()), 3))
print("relevant_docs_per_query (subset) - median:", round(float(relevant_counts.median()), 3))


## Build Embeddings and Inspect Shapes

Expected shapes:
- documents: `[n_documents, embedding_dim]`
- queries: `[n_queries, embedding_dim]`


In [ ]:
baseline_device = _resolve_embedding_device_spec(DEFAULT_EMBEDDING_DEVICE)

t0 = time.perf_counter()
doc_emb, query_emb = build_embeddings(
    docs,
    queries,
    text_field=TEXT_FIELD,
    model_name=MODEL_NAME,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    cache_dir=CACHE_DIR,
    device=baseline_device,
    local_files_only=LOCAL_FILES_ONLY,
)
embedding_time = time.perf_counter() - t0

print("doc_emb shape:", doc_emb.shape)
print("query_emb shape:", query_emb.shape)
print(f"Embedding build time: {embedding_time:.3f}s")
print("baseline_device:", baseline_device or "auto")
print("local_files_only:", LOCAL_FILES_ONLY)


## 2D Visualization (UMAP/t-SNE)

A 2D projection is useful to inspect whether semantically related texts are grouped together.
The plot below colors points by category and uses different markers for documents and queries.


In [ ]:
# Build a projection subset for speed and readability.
rng = np.random.default_rng(42)
n_docs_plot = min(PLOT_MAX_DOCS, len(docs))

if len(docs) > n_docs_plot:
    plot_doc_idx = np.sort(rng.choice(len(docs), size=n_docs_plot, replace=False))
else:
    plot_doc_idx = np.arange(len(docs))

doc_emb_plot = doc_emb[plot_doc_idx]
docs_plot = [docs[i] for i in plot_doc_idx]

all_emb = np.vstack([doc_emb_plot, query_emb])
doc_labels = [str(d.get("category", "unknown")) for d in docs_plot]
query_labels = [str(q.get("category", "unknown")) for q in queries]
all_labels = doc_labels + query_labels
all_types = ["doc"] * len(docs_plot) + ["query"] * len(queries)

# Prefer UMAP when available; fallback to t-SNE.
try:
    import umap
    reducer = umap.UMAP(n_neighbors=12, min_dist=0.15, random_state=42)
    proj = reducer.fit_transform(all_emb)
    method_name = "UMAP"
except Exception:
    from sklearn.manifold import TSNE
    perplexity = min(30, max(5, (all_emb.shape[0] - 1) // 3))
    reducer = TSNE(n_components=2, perplexity=perplexity, random_state=42)
    proj = reducer.fit_transform(all_emb)
    method_name = "t-SNE"

unique_labels = sorted(set(all_labels))
cmap = plt.get_cmap("tab20", len(unique_labels))
color_map = {label: cmap(i) for i, label in enumerate(unique_labels)}

plt.figure(figsize=(9, 7))
for i, (x, y) in enumerate(proj):
    c = color_map.get(all_labels[i], "gray")
    marker = "o" if all_types[i] == "doc" else "x"
    alpha = 0.7 if all_types[i] == "doc" else 1.0
    size = 20 if all_types[i] == "doc" else 70
    plt.scatter(x, y, c=[c], marker=marker, alpha=alpha, s=size)

plt.title(f"Embeddings projected in 2D ({method_name})")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.grid(alpha=0.2)
plt.show()


### Interpretation

Points from the same semantic category tend to be closer after projection.
This is expected because embeddings encode semantic similarity, and category-specific vocabulary pushes related texts to nearby regions.


## Retrieval with Cosine Similarity

For each query:
1. Compute cosine similarity with all document vectors.
2. Keep the top-k document indices and scores.
3. Map indices to document IDs for readability.


In [ ]:
# Keep only valid k values for the document set size.
k_eval = [k for k in K_VALUES if k <= len(docs)]
if not k_eval:
    raise ValueError("No valid k for current number of documents.")

doc_ids = [str(d["id"]) for d in docs]

retrieval_rows = []
last_topk_indices = None
last_topk_scores = None

for k in k_eval:
    t1 = time.perf_counter()
    topk_indices, topk_scores = retrieve_embeddings(doc_emb, query_emb, k=k)
    retrieval_time = time.perf_counter() - t1

    # Convert indices to IDs for evaluation/inspection.
    topk_docids = [[doc_ids[idx] for idx in row] for row in topk_indices]
    metrics = evaluate_run(topk_docids, gt_eval, query_ids, k)
    phase1_score = (metrics["precision@k"] + metrics["recall@k"] + metrics["mrr@k"]) / 3.0

    retrieval_rows.append({
        "k": k,
        "retrieval_time_s": retrieval_time,
        "precision@k": metrics["precision@k"],
        "recall@k": metrics["recall@k"],
        "mrr@k": metrics["mrr@k"],
        "phase1_score": phase1_score,
    })

    print(
        f"k={k} | retrieval_time={retrieval_time:.4f}s | "
        f"P={metrics['precision@k']:.4f} R={metrics['recall@k']:.4f} "
        f"MRR={metrics['mrr@k']:.4f} phase1={phase1_score:.4f}"
    )
    print("Q1 top doc_ids:", topk_docids[0][:min(5, k)])

    last_topk_indices = topk_indices
    last_topk_scores = topk_scores

retrieval_df = pd.DataFrame(retrieval_rows)
retrieval_df


## Output Contract Check

The retrieval outputs follow the required contract:
- `topk_indices`: shape `[n_queries, k]`
- `topk_scores`: shape `[n_queries, k]`


In [ ]:
# Show the shape contract for the last evaluated k.
print("topk_indices shape:", None if last_topk_indices is None else last_topk_indices.shape)
print("topk_scores shape:", None if last_topk_scores is None else last_topk_scores.shape)

if last_topk_indices is not None:
    print("Sample indices row (Q1):", last_topk_indices[0][:10])
if last_topk_scores is not None:
    print("Sample scores row (Q1):", np.round(last_topk_scores[0][:10], 4))


## Configuration Benchmark on Train Subset

This section runs multiple embedding configurations on the current train subset (`docs`, `queries`) and reports:
- retrieval quality (`Precision@k`, `Recall@k`, `MRR@k`, and Phase-1 average score)
- build time (embedding generation)
- retrieval time (top-k search)
- total time (build + retrieval)


In [ ]:
if not EMBEDDING_EXPERIMENTS:
    raise ValueError("EMBEDDING_EXPERIMENTS is empty. Add at least one config.")

k_eval = [k for k in K_VALUES if k <= len(docs)]
if not k_eval:
    raise ValueError("No valid k for current number of documents.")
if PRIMARY_K not in k_eval:
    PRIMARY_K = max(k_eval)
    print(f"PRIMARY_K adjusted to {PRIMARY_K} (largest valid k).")

doc_ids = [str(d["id"]) for d in docs]
rows = []

def _cfg_value(cfg: dict, key: str, default):
    return cfg[key] if key in cfg else default

for exp in EMBEDDING_EXPERIMENTS:
    name = str(_cfg_value(exp, "name", _cfg_value(exp, "model_name", "exp")))
    model_name = str(_cfg_value(exp, "model_name", MODEL_NAME))
    batch_size = int(_cfg_value(exp, "batch_size", BATCH_SIZE))
    requested_device = _cfg_value(exp, "device", DEFAULT_EMBEDDING_DEVICE)
    device = _resolve_embedding_device_spec(requested_device)
    precision = str(_cfg_value(exp, "precision", "float32"))
    model_max_seq_length = _cfg_value(exp, "model_max_seq_length", None)
    truncate_dim = _cfg_value(exp, "truncate_dim", None)
    chunk_size = _cfg_value(exp, "chunk_size", None)
    normalize_embeddings = bool(_cfg_value(exp, "normalize_embeddings", True))
    local_files_only = bool(_cfg_value(exp, "local_files_only", LOCAL_FILES_ONLY))

    t_embed0 = time.perf_counter()
    doc_emb_exp, query_emb_exp = build_embeddings(
        docs,
        queries,
        text_field=TEXT_FIELD,
        model_name=model_name,
        batch_size=batch_size,
        show_progress_bar=BENCHMARK_SHOW_PROGRESS_BAR,
        cache_dir=CACHE_DIR,
        device=device,
        precision=precision,
        model_max_seq_length=model_max_seq_length,
        truncate_dim=truncate_dim,
        chunk_size=chunk_size,
        normalize_embeddings=normalize_embeddings,
        local_files_only=local_files_only,
    )
    embedding_time_s = time.perf_counter() - t_embed0

    print(
        f"[{name}] emb_time={embedding_time_s:.2f}s | model={model_name} | "
        f"batch={batch_size} | device={device or 'auto'} | local_only={local_files_only}"
    )

    for k in k_eval:
        t_ret0 = time.perf_counter()
        topk_indices, _ = retrieve_embeddings(doc_emb_exp, query_emb_exp, k=k)
        retrieval_time_s = time.perf_counter() - t_ret0

        pred_docids = [[doc_ids[idx] for idx in row] for row in topk_indices]
        metrics = evaluate_run(pred_docids, gt_eval, query_ids, k)
        phase1_score = (metrics["precision@k"] + metrics["recall@k"] + metrics["mrr@k"]) / 3.0

        rows.append({
            "experiment": name,
            "model_name": model_name,
            "batch_size": batch_size,
            "device": device,
            "precision": precision,
            "model_max_seq_length": model_max_seq_length,
            "truncate_dim": truncate_dim,
            "chunk_size": chunk_size,
            "normalize_embeddings": normalize_embeddings,
            "local_files_only": local_files_only,
            "k": k,
            "embedding_time_s": embedding_time_s,
            "retrieval_time_s": retrieval_time_s,
            "total_time_s": embedding_time_s + retrieval_time_s,
            "precision@k": metrics["precision@k"],
            "recall@k": metrics["recall@k"],
            "mrr@k": metrics["mrr@k"],
            "phase1_score": phase1_score,
        })

benchmark_df = pd.DataFrame(rows)
if benchmark_df.empty:
    raise RuntimeError("Benchmark produced no rows.")

benchmark_primary = (
    benchmark_df[benchmark_df["k"] == PRIMARY_K]
    .sort_values(["phase1_score", "total_time_s"], ascending=[False, True])
    .reset_index(drop=True)
)

print(f"\n=== Leaderboard at k={PRIMARY_K} (quality first) ===")
benchmark_primary[
    [
        "experiment",
        "phase1_score",
        "precision@k",
        "recall@k",
        "mrr@k",
        "embedding_time_s",
        "retrieval_time_s",
        "total_time_s",
    ]
]

print(f"\n=== Fastest at k={PRIMARY_K} (time first) ===")
benchmark_primary.sort_values("total_time_s", ascending=True).head(10)[
    [
        "experiment",
        "total_time_s",
        "embedding_time_s",
        "retrieval_time_s",
        "phase1_score",
    ]
]


## RRF4 Sweep on Train (4-source candidates, no CE)

### How it works
1. Build candidate rankings from **4 complementary sources**:
- embeddings from `L12_quality`
- embeddings from `L12_balanced_128`
- BM25 Plus
- BM25 Okapi
2. For each config in `RRF4_SWEEP_CONFIGS`:
- set `candidate_k` via `candidate_multiplier`
- fuse the 4 rankings with weighted **RRF** (`rrf_k` + per-source weights)
- evaluate Precision/Recall/MRR and `phase1_score` on train.

### Why this section
- You can test many RRF4 configurations automatically on train.
- You can compare quality and runtime before using notebook 06 for submission.


In [ ]:
import re
from rank_bm25 import BM25Okapi, BM25Plus


def _tokenize_bm25(text: str) -> list[str]:
    return re.findall(r"\b[a-zA-Z0-9]+\b", str(text).lower())


def _fit_bm25_model(docs_local: list[dict], text_field: str, method: str):
    tokenized = [_tokenize_bm25(d.get(text_field, "")) for d in docs_local]
    if method == "plus":
        return BM25Plus(tokenized)
    if method == "okapi":
        return BM25Okapi(tokenized)
    raise ValueError("bm25 method must be 'plus' or 'okapi'.")


def _retrieve_bm25_indices(bm25_model, queries_local: list[dict], k: int, text_field: str) -> np.ndarray:
    out = np.zeros((len(queries_local), k), dtype=np.int64)
    for i, q in enumerate(queries_local):
        scores = np.asarray(bm25_model.get_scores(_tokenize_bm25(q.get(text_field, ""))))
        topk = np.argsort(-scores)[:k]
        out[i, :len(topk)] = topk
    return out


def _pick_embedding_config(exp_name: str, experiments: list[dict]) -> dict:
    for exp in experiments:
        if str(exp.get("name")) == exp_name:
            return exp
    raise ValueError(f"Unknown embedding experiment '{exp_name}'")


def _retrieve_embedding_indices(
    docs_local: list[dict],
    queries_local: list[dict],
    cfg: dict,
    candidate_k: int,
) -> tuple[np.ndarray, float]:
    t0 = time.perf_counter()
    requested_device = cfg.get("device", DEFAULT_EMBEDDING_DEVICE)
    device = _resolve_embedding_device_spec(requested_device)
    local_files_only = bool(cfg.get("local_files_only", LOCAL_FILES_ONLY))
    doc_emb, query_emb = build_embeddings(
        docs_local,
        queries_local,
        text_field=TEXT_FIELD,
        model_name=str(cfg.get("model_name", MODEL_NAME)),
        batch_size=int(cfg.get("batch_size", BATCH_SIZE)),
        show_progress_bar=BENCHMARK_SHOW_PROGRESS_BAR,
        cache_dir=CACHE_DIR,
        device=device,
        precision=str(cfg.get("precision", "float32")),
        model_max_seq_length=cfg.get("model_max_seq_length", None),
        truncate_dim=cfg.get("truncate_dim", None),
        chunk_size=cfg.get("chunk_size", None),
        normalize_embeddings=bool(cfg.get("normalize_embeddings", True)),
        local_files_only=bool(cfg.get("local_files_only", LOCAL_FILES_ONLY)),
    )
    topk_indices, _ = retrieve_embeddings(doc_emb, query_emb, k=candidate_k)
    return topk_indices, time.perf_counter() - t0


def _rrf_fuse_named(
    rankings_by_source: dict[str, np.ndarray],
    source_order: list[str],
    weights_by_source: dict[str, float],
    k_out: int,
    rrf_k: int,
) -> np.ndarray:
    nq = rankings_by_source[source_order[0]].shape[0]
    fused = np.zeros((nq, k_out), dtype=np.int64)

    for qi in range(nq):
        score_map: dict[int, float] = {}
        for src_name in source_order:
            arr = rankings_by_source[src_name]
            w = float(weights_by_source[src_name])
            for rank, didx in enumerate(arr[qi], start=1):
                d = int(didx)
                score_map[d] = score_map.get(d, 0.0) + (w / (rrf_k + rank))

        ranked = sorted(score_map.items(), key=lambda x: x[1], reverse=True)
        fused[qi, :k_out] = [d for d, _ in ranked[:k_out]]

    return fused


def _resolve_weights(cfg: dict, sources: list[dict]) -> dict[str, float]:
    default_weights = {str(s["name"]): float(s.get("weight", 1.0)) for s in sources}
    override = {str(k): float(v) for k, v in dict(cfg.get("weights", {})).items()}

    unknown = sorted(set(override.keys()).difference(default_weights.keys()))
    if unknown:
        raise ValueError(f"Unknown weight keys in config '{cfg.get('name')}': {unknown}")

    out = default_weights.copy()
    out.update(override)

    for src_name, w in out.items():
        if w <= 0:
            raise ValueError(f"Source weight must be > 0 for '{src_name}'")

    return out


if not RERANK_CANDIDATE_SOURCES:
    raise ValueError("RERANK_CANDIDATE_SOURCES is empty.")
if not RRF4_SWEEP_CONFIGS:
    raise ValueError("RRF4_SWEEP_CONFIGS is empty.")

k_eval = [k for k in K_VALUES if k <= len(docs)]
if not k_eval:
    raise ValueError("No valid k values for current docs subset.")

if PRIMARY_K not in k_eval:
    PRIMARY_K = max(k_eval)
    print(f"PRIMARY_K adjusted to {PRIMARY_K} (largest valid k).")

max_k_needed = max(k_eval)
source_order = [str(s["name"]) for s in RERANK_CANDIDATE_SOURCES]

sweep_meta = []
for cfg in RRF4_SWEEP_CONFIGS:
    name = str(cfg.get("name", f"rrf4_cfg_{len(sweep_meta)}"))
    mult = int(cfg.get("candidate_multiplier", 1))
    if mult <= 0:
        raise ValueError(f"candidate_multiplier must be > 0 in config '{name}'")

    candidate_k = min(len(docs), max(max_k_needed, max_k_needed * mult))
    rrf_k = int(cfg.get("rrf_k", 20))
    if rrf_k <= 0:
        raise ValueError(f"rrf_k must be > 0 in config '{name}'")

    sweep_meta.append(
        {
            "name": name,
            "candidate_multiplier": mult,
            "candidate_k": candidate_k,
            "rrf_k": rrf_k,
            "weights": _resolve_weights(cfg, RERANK_CANDIDATE_SOURCES),
        }
    )

max_candidate_k = max(int(m["candidate_k"]) for m in sweep_meta)
print(f"n_rrf4_configs={len(sweep_meta)} | max_candidate_k={max_candidate_k}")

# ---- Build source rankings once at max_candidate_k (shared cost) ----
rankings_max: dict[str, np.ndarray] = {}
source_rows: list[dict] = []

for src in RERANK_CANDIDATE_SOURCES:
    src_name = str(src.get("name", "source"))
    src_type = str(src.get("type", "embedding")).lower()

    t0 = time.perf_counter()
    if src_type == "embedding":
        exp_name = str(src.get("embedding_experiment"))
        cfg = _pick_embedding_config(exp_name, EMBEDDING_EXPERIMENTS)
        idx_max, _ = _retrieve_embedding_indices(docs, queries, cfg, max_candidate_k)

    elif src_type == "bm25":
        method = str(src.get("bm25_method", "plus"))
        bm25_model = _fit_bm25_model(docs, text_field=TEXT_FIELD, method=method)
        idx_max = _retrieve_bm25_indices(
            bm25_model,
            queries,
            k=max_candidate_k,
            text_field=TEXT_FIELD,
        )

    else:
        raise ValueError(f"Unsupported source type: {src_type}")

    elapsed = time.perf_counter() - t0
    rankings_max[src_name] = idx_max
    source_rows.append(
        {
            "source": src_name,
            "type": src_type,
            "build_time_s": elapsed,
            "k_built": max_candidate_k,
        }
    )

source_df = pd.DataFrame(source_rows)
shared_source_build_time_s = float(source_df["build_time_s"].sum())
print("=== Shared source build timings (built once) ===")
source_df

print(f"shared_source_build_time_s={shared_source_build_time_s:.2f}")

# ---- RRF4 sweep ----
id_by_index = [str(d.get("id")) for d in docs]
sweep_rows: list[dict] = []

for meta in sweep_meta:
    cfg_name = str(meta["name"])
    candidate_k = int(meta["candidate_k"])
    rrf_k = int(meta["rrf_k"])
    weights = dict(meta["weights"])

    cfg_t0 = time.perf_counter()

    rankings_cfg = {src_name: rankings_max[src_name][:, :candidate_k] for src_name in source_order}

    fused_indices = _rrf_fuse_named(
        rankings_by_source=rankings_cfg,
        source_order=source_order,
        weights_by_source=weights,
        k_out=candidate_k,
        rrf_k=rrf_k,
    )
    candidate_docids = [[id_by_index[idx] for idx in row] for row in fused_indices]

    for k in k_eval:
        m = evaluate_run(candidate_docids, gt_eval, query_ids, k)
        phase1 = (m["precision@k"] + m["recall@k"] + m["mrr@k"]) / 3.0

        sweep_rows.append(
            {
                "config": cfg_name,
                "k": k,
                "candidate_multiplier": int(meta["candidate_multiplier"]),
                "candidate_k": candidate_k,
                "rrf_k": rrf_k,
                "w_emb_L12_quality": float(weights["emb_L12_quality"]),
                "w_emb_L12_balanced": float(weights["emb_L12_balanced"]),
                "w_bm25_plus": float(weights["bm25_plus"]),
                "w_bm25_okapi": float(weights["bm25_okapi"]),
                "precision@k": m["precision@k"],
                "recall@k": m["recall@k"],
                "mrr@k": m["mrr@k"],
                "phase1_score": phase1,
            }
        )

    cfg_runtime_s = time.perf_counter() - cfg_t0
    for row in sweep_rows[-len(k_eval):]:
        row["config_runtime_s"] = cfg_runtime_s
        row["shared_source_build_time_s"] = shared_source_build_time_s
        row["total_time_s"] = shared_source_build_time_s + cfg_runtime_s

    print(
        f"[{cfg_name}] candidate_k={candidate_k} rrf_k={rrf_k} "
        f"config_runtime_s={cfg_runtime_s:.2f}"
    )

rrf4_sweep_df = pd.DataFrame(sweep_rows)
if rrf4_sweep_df.empty:
    raise RuntimeError("RRF4 sweep produced no rows.")

rrf4_primary = (
    rrf4_sweep_df[rrf4_sweep_df["k"] == PRIMARY_K]
    .sort_values(["phase1_score", "total_time_s"], ascending=[False, True])
    .reset_index(drop=True)
)

print(f"\n=== RRF4 Leaderboard at k={PRIMARY_K} (quality first) ===")
rrf4_primary[
    [
        "config",
        "phase1_score",
        "precision@k",
        "recall@k",
        "mrr@k",
        "candidate_k",
        "rrf_k",
        "w_emb_L12_quality",
        "w_emb_L12_balanced",
        "w_bm25_plus",
        "w_bm25_okapi",
        "total_time_s",
        "shared_source_build_time_s",
        "config_runtime_s",
    ]
]


## Next Step

1. Keep the top config(s) from `rrf4_primary` at `k=PRIMARY_K`.
2. Re-run the best 1-3 configs with any final tweaks to weights or `rrf_k`.
3. Transfer the winner to `notebooks/06_phase1_submission.ipynb` for final Kaggle submission.
